<a href="https://colab.research.google.com/github/Gegee-notavailable/thai-fake-news-detection-TFSVM/blob/main/fake_news_typhoon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers accelerate bitsandbytes huggingface_hub scikit-learn tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 57.9 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.31.0
    Uninstalling huggingface_hub-1.31.0:
      Successfully uninstalled huggingface_hub-1.31.0
  Attempti

In [2]:
import pandas as pd
import torch
from huggingface_hub import notebook_login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

# 1. ล็อกอิน Hugging Face
notebook_login()

# 2. โหลดชุดข้อมูล 2 ไฟล์
file_list = [
    "AFNC_Opendata_export_20260930221100(2025).csv",
    "AFNC_Opendata_export_20260930221100(2026).csv"
]
print("กำลังโหลดชุดข้อมูลทดสอบ...")
df_list = [pd.read_csv(f, encoding='utf-8-sig') for f in file_list]
df_external = pd.concat(df_list, ignore_index=True)
df_external = df_external[['หัวข้อข่าว', 'ประเภทข่าว']].dropna()

# 3. โหลดโมเดล Typhoon LLM (แบบ 4-bit บน GPU ตัวแรก)
model_id = "typhoon-ai/typhoon2-qwen2.5-7b"
print("กำลังโหลดโมเดล Typhoon LLM...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map={'': 0}
)

# 4. ฟังก์ชันทำนายผล
def predict_with_typhoon(title):
    prompt = f"""คุณคือนักตรวจสอบข้อเท็จจริงข่าวกรอง จงวิเคราะห์หัวข้อข่าวต่อไปนี้ว่าเป็น "ข่าวจริง" หรือ "ข่าวปลอม"
ตอบสั้นๆ ว่า "ข่าวจริง" หรือ "ข่าวปลอม" เท่านั้น ห้ามอธิบายเพิ่ม

หัวข้อข่าว: {title}
คำตอบ:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    ans = response.split("คำตอบ:")[-1].strip()
    return 1 if "ข่าวจริง" in ans else 0

# 5. รันประเมินผล
true_labels = []
pred_labels = []

print("เริ่มรันประเมินผลด้วย Typhoon LLM...")
for idx, row in tqdm(df_external.iterrows(), total=len(df_external)):
    title = row['หัวข้อข่าว']
    actual_status = 1 if str(row['ประเภทข่าว']).strip() == 'ข่าวจริง' else 0

    predicted_status = predict_with_typhoon(title)

    true_labels.append(actual_status)
    pred_labels.append(predicted_status)

# 6. คำนวณ Metrics สรุปผล
accuracy = accuracy_score(true_labels, pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary', zero_division=0)

print("\n--- ผลการทดสอบ Typhoon LLM กับชุดข้อมูลภายนอก ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

กำลังโหลดชุดข้อมูลทดสอบ...
กำลังโหลดโมเดล Typhoon LLM...


config.json:   0%|          | 0.00/626 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

เริ่มรันประเมินผลด้วย Typhoon LLM...


100%|██████████| 259/259 [04:06<00:00,  1.05it/s]


--- ผลการทดสอบ Typhoon LLM กับชุดข้อมูลภายนอก ---
Accuracy: 0.3784
Precision: 0.3798
Recall: 0.7117
F1-Score: 0.4953
